# Pertemuan 11 — Lab Starter: Transformer & Attention
Isi bagian bertanda `____` (TODO). Jalankan sel dari atas ke bawah.
- **Bagian A (10 mnt)**: attention dari nol, NumPy saja.
- **Bagian B (15 mnt)**: kantong kata vs model pre-trained. Jalankan di **Google Colab** (butuh internet).
- **Bagian C (12 mnt)**: LLM lokal lewat Ollama. Paling mudah di **Google Colab** juga: jalankan sel C0 dulu, ia memasang Ollama dan mengunduh model kecil.

Semua bagian bisa dikerjakan sendiri di Colab. Mulai dari sel Bagian 0a untuk mengunggah dua CSV.

In [3]:
# BAGIAN 0a (HANYA di Google Colab): unggah dua berkas CSV. Klik 'Choose Files', pilih keduanya.
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("Bukan Colab: pastikan dua CSV ada di folder yang sama dengan notebook ini.")

Saving ulasan_latih_600.csv to ulasan_latih_600.csv
Saving ulasan_uji_48.csv to ulasan_uji_48.csv


In [4]:
# BAGIAN 0 (sudah disediakan): data
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
latih = pd.read_csv("ulasan_latih_600.csv")
uji   = pd.read_csv("ulasan_uji_48.csv")
print(latih.shape, uji.shape)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

(600, 3) (48, 4)


## Bagian A — attention dari nol
Target: bobot 'bersih' ke 'tidak' = 0.807.

In [ ]:
# BAGIAN A (10 mnt): attention dari nol untuk "kamar tidak bersih"
kata = ["kamar", "tidak", "bersih"]
X  = np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])
Wq = np.array([[0., 1.], [0., 0.], [1., 0.]])
Wk = np.array([[0., 1.], [3., 0.], [0., 1.]])
Wv = np.array([[0., 0.], [0., 1.], [1., 0.]])
Q, K, V = X @ Wq, X @ Wk, X @ Wv          # TODO-1: X @ Wq, X @ Wk, X @ Wv
skala = (Q @ K.T) / np.sqrt(K.shape[1])   # TODO-2: Q @ K.T
A = softmax(skala)                        # TODO-3: softmax(skala)
Z = A @ V                                 # TODO-4: A @ V
print("A =\n", A); print("Z =\n", Z)
print("bobot 'bersih' ke 'tidak':", round(float(A[2, 1]), 3))

A =
 [[0.4011 0.1978 0.4011]
 [0.3333 0.3333 0.3333]
 [0.0967 0.8066 0.0967]]
Z =
 [[0.4011 0.1978]
 [0.3333 0.3333]
 [0.0967 0.8066]]
bobot 'bersih' ke 'tidak': 0.807


In [ ]:
# Cek mandiri Bagian A
ok = abs(float(A[2, 1]) - 0.8066) < 0.001
print("BENAR: bobot 'bersih' ke 'tidak' =", round(float(A[2, 1]), 3) if ok else "")
if not ok:
    print("Belum cocok. Cek: K ditranspos (K.T)? sudah dibagi akar d_k? softmax per baris?")

BENAR: bobot 'bersih' ke 'tidak' = 0.807


## Bagian B — kantong kata vs pre-trained
Isi tabel di slide dengan hasil `akurasi`.

In [5]:
# BAGIAN B (15 mnt): kantong kata vs model pre-trained (Colab)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
vek = TfidfVectorizer()
svm = LinearSVC(random_state=7).fit(vek.fit_transform(latih["ulasan"]), latih["label"])
uji["tebak_bow"] = svm.predict(vek.transform(uji["ulasan"]))
# !pip install -q transformers torch
from transformers import pipeline
model = "w11wo/indonesian-roberta-base-sentiment-classifier"
pembaca = pipeline("text-classification", model=model, top_k=None)
def pilih(hasil):          # bandingkan positive vs negative
    skor = {h["label"]: h["score"] for h in hasil}
    return "positif" if skor["positive"] >= skor["negative"] else "negatif"
hasil = pembaca(list(uji["ulasan"]))
uji["tebak_tf"] = [pilih(h) for h in hasil]       # TODO-5: [pilih(h) for h in hasil].
tabel = uji.assign(bow=uji["tebak_bow"] == uji["label"],
                   tf=uji["tebak_tf"] == uji["label"])
print(tabel.groupby("jenis")[["bow", "tf"]].mean().round(3))   # TODO-6: jenis
# biasa    1.000  1.000
# gaul     0.750  1.000
# kontras  0.750  0.917
# negasi   0.417  1.000

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

           bow     tf
jenis                
biasa    1.000  1.000
gaul     0.750  1.000
kontras  0.750  0.917
negasi   0.417  1.000


In [6]:
# Menampilkan ulasan yang salah diprediksi oleh BoW atau Pre-trained
salah = uji[
    (uji["tebak_bow"] != uji["label"]) |
    (uji["tebak_tf"] != uji["label"])
]

display(salah[["ulasan", "label", "tebak_bow", "tebak_tf"]].head(3))

,ulasan,label,tebak_bow,tebak_tf
13,"Stafnya tidak pernah mengecewakan, selalu siga...",positif,negatif,positif
14,"Tidak ada keluhan, semuanya berjalan lancar.",positif,negatif,positif
15,"Sarapannya tidak mengecewakan, pilihannya bany...",positif,negatif,positif


## Bagian C — LLM lokal
Di Colab: jalankan sel C0 dulu (3-6 menit). Di laptop sendiri yang sudah ada Ollama: lewati C0. Model awal `qwen2.5:1.5b`; ganti sesuai `ollama list` bila perlu. Lalu isi TODO-7 dan TODO-8.

In [ ]:
# BAGIAN C0 (HANYA di Google Colab): pasang Ollama dan unduh model kecil. Sekali jalan, sekitar 3-6 menit.
import subprocess, time, shutil
if shutil.which("ollama") is None:
    subprocess.run("apt-get -qq install -y zstd", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
subprocess.run(["ollama", "pull", "qwen2.5:1.5b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

NAME            ID              SIZE      MODIFIED               
qwen2.5:1.5b    65ec06548149    986 MB    Less than a second ago    



In [ ]:
# BAGIAN C (12 mnt): LLM lokal (Ollama) pada 12 ulasan negasi
import requests
OLLAMA, MODEL = "http://localhost:11434", "qwen2.5:1.5b"
def tanya(ulasan, contoh=""):
    perintah = ("Kamu penilai ulasan hotel. Jawab HANYA satu kata: "
                "positif atau negatif.\n" + contoh +
                "Ulasan: " + ulasan + "\nJawaban:")
    r = requests.post(OLLAMA + "/api/chat", timeout=120, json={
        "model": MODEL, "stream": False, "options": {"temperature": 0},
        "messages": [{"role": "user", "content": perintah}]})
    return r.json()["message"]["content"].strip().lower()
negasi = uji[uji["jenis"] == "negasi"]
tebak = [tanya(u) for u in negasi["ulasan"]]          # TODO-7: tebak = [tanya(u) for u in negasi["ulasan"]]
cocok = [t.startswith(l) for t, l in zip(tebak, negasi["label"])]
print("akurasi LLM (negasi):", sum(cocok), "dari", len(cocok))   # TODO-8: coba prompt few-shot

akurasi LLM (negasi): 12 dari 12


In [ ]:

# FEW-SHOT: memberikan dua contoh dari data latih

contoh_positif = latih[latih["label"] == "positif"]["ulasan"].iloc[0]
contoh_negatif = latih[latih["label"] == "negatif"]["ulasan"].iloc[0]

contoh = (
    "Contoh:\n"
    f"Ulasan: {contoh_positif}\n"
    "Jawaban: positif\n\n"
    f"Ulasan: {contoh_negatif}\n"
    "Jawaban: negatif\n\n"
)

# Prediksi ulang 12 ulasan negasi dengan contoh
tebak_few = [tanya(u, contoh) for u in negasi["ulasan"]]

cocok_few = [
    t.startswith(l)
    for t, l in zip(tebak_few, negasi["label"])
]

print("Akurasi LLM few-shot:",
      sum(cocok_few), "dari", len(cocok_few))


Akurasi LLM few-shot: 12 dari 12


In [ ]:

# Menampilkan ulasan yang salah diprediksi
salah = uji[
    (uji["tebak_bow"] != uji["label"]) |
    (uji["tebak_tf"] != uji["label"])
]

display(salah[["ulasan", "label", "tebak_bow", "tebak_tf"]].head(3))


,ulasan,label,tebak_bow,tebak_tf
13,"Stafnya tidak pernah mengecewakan, selalu siga...",positif,negatif,positif
14,"Tidak ada keluhan, semuanya berjalan lancar.",positif,negatif,positif
15,"Sarapannya tidak mengecewakan, pilihannya bany...",positif,negatif,positif
